In [1]:
from pathlib import Path
import csv
import json
import re
import sys

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display

# Change PROJECT_DIR if the notebook is NOT stored in TIME_VARY_50N.
PROJECT_DIR = Path.cwd().resolve()
TOPOLOGY_PATTERN = "50_CNs_topology_*.json"
CHECKPOINT_PATH = PROJECT_DIR / "ha2c_trained_models_50_nodes_lr_0_00001.pt"
OUTPUT_PREFIX = PROJECT_DIR / "ha2c_v8_50_topology_predictions"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Same constants as the attached 50-node V8 training notebook.
EXPECTED_NODES = 50
MAX_STEPS = 200
LARGE_VALUE = 20.0
LATENCY_LIMIT_MS = 10.0   # 10 ms is allowed; Agent 2 may add 0 ms.
FIXED_COSTS = np.array([5.0, 10.0, 20.0])
VARIABLE_COSTS_1 = np.array([[4.0, 8.0, 16.0], [1.0, 2.0, 4.0]])
VARIABLE_COSTS_2 = np.array([[2.0, 4.0, 8.0], [2.0, 4.0, 8.0]])

print(f"Project: {PROJECT_DIR}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Inference device: {DEVICE}")

Project: /home/cci/Downloads/Latest/Revised/Time_Vary_50N
Checkpoint: /home/cci/Downloads/Latest/Revised/Time_Vary_50N/ha2c_trained_models_50_nodes_lr_0_00001.pt
Inference device: cuda


## 1. Actor architecture and hierarchical V8 inference helpers

These reproduce the attached V8 model shape, action masking, placement-cost sharing, first-stage association, Agent-2 state construction, and end-to-end latency verification. The saved critic and optimizer weights are not needed for prediction.

In [2]:
class Actor(nn.Module):
    """Exact CNN/FC architecture of Actor in the uploaded V8 notebook."""

    def __init__(self, num_nodes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * (num_nodes // 8) * (num_nodes // 8), 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc4 = nn.Linear(512, num_nodes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x.unsqueeze(1))))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc4(x)  # Raw logits: same logits fed into V8 Categorical.


def load_actors(checkpoint_path, device):
    # Load only trained actor weights: no critic/optimizer is needed for inference.
    # Only load checkpoints you created/trust. Fallback is for older PyTorch.
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location=device)

    actors = []
    for key in ("actor1_state_dict", "actor2_state_dict"):
        if key not in checkpoint:
            raise KeyError(f"Missing {key!r} in checkpoint: {checkpoint_path}")
        model = Actor(EXPECTED_NODES).to(device)
        model.load_state_dict(checkpoint[key], strict=True)
        model.eval()
        actors.append(model)
    return actors


def node_classes_for_topology(cloud_nodes, regional_nodes, edge_nodes, n):
    """Reproduce encode_node_types() from V8, using node IDs as matrix indices."""
    types = {}
    for node_group, class_id in ((cloud_nodes, 0), (regional_nodes, 1), (edge_nodes, 2)):
        for node_id in np.atleast_1d(node_group).astype(int).tolist():
            if node_id in types:
                raise ValueError(f"Topology node {node_id} appears in multiple tiers")
            types[node_id] = class_id
    if set(types) != set(range(n)):
        raise ValueError(f"Expected node IDs 0..{n - 1}; found {sorted(types)}")
    return types


def new_action_cost(node_id, already_activated, variable_costs, node_classes):
    """Exactly V8 compute_cost_new(): fixed once; current agent components per action."""
    node_class = node_classes[int(node_id)]
    fixed = 0.0 if int(node_id) in already_activated else FIXED_COSTS[node_class]
    variable = float(np.sum(variable_costs[:, node_class]))
    return float(fixed + variable)


@torch.no_grad()
def masked_policy_action(actor, state, valid_actions, device):
    """Deterministic trained-policy prediction, subject to V8 valid-action mask.

    V8 training uses Categorical(logits=masked_logits).sample(). For reproducible
    inference, use the argmax of the same masked logits. No optimizer updates.
    """
    policy_logits = actor(
        torch.from_numpy(np.asarray(state, dtype=np.float32)).unsqueeze(0).to(device)
    ).squeeze(0)
    if not torch.isfinite(policy_logits).all():
        raise RuntimeError("Actor produced NaN/Inf logits on this topology")
    allowed = torch.as_tensor(valid_actions, dtype=torch.bool, device=device)
    if not bool(allowed.any().item()):
        raise RuntimeError("No valid action remains; cannot predict a complete placement")
    masked_logits = policy_logits.masked_fill(~allowed, torch.finfo(policy_logits.dtype).min)
    chosen = int(torch.argmax(masked_logits).item())
    if not valid_actions[chosen]:
        raise RuntimeError(f"Action masking failed for predicted node {chosen}")
    return chosen


def select_nodes(actor, initial_state, target_nodes, variable_costs,
                 node_classes, device, preactivated_nodes=()):
    """V8 sequential state transition, mask, and cost accounting for one agent."""
    working = np.asarray(initial_state, dtype=np.float64).copy()
    targets = np.asarray(target_nodes, dtype=int)
    chosen_nodes = []
    accumulated_cost = 0.0
    activated = set(int(i) for i in preactivated_nodes)
    if len(targets) == 0:
        raise ValueError("Empty target set")

    # A target is 'covered' once a selected row has a feasible link to it.
    remaining = set(targets.tolist())
    for _ in range(MAX_STEPS):
        if not remaining:
            break
        valid_actions = np.any(working[:, targets] <= LATENCY_LIMIT_MS, axis=1)
        if not np.any(valid_actions):
            raise RuntimeError(f"Uncovered target nodes with no valid action: {sorted(remaining)}")
        node_id = masked_policy_action(actor, working, valid_actions, device)
        newly_covered = targets[working[node_id, targets] <= LATENCY_LIMIT_MS]
        newly_covered = [int(t) for t in newly_covered if int(t) in remaining]
        if not newly_covered:
            raise RuntimeError(f"Predicted node {node_id} covers no new target")

        accumulated_cost += new_action_cost(node_id, activated, variable_costs, node_classes)
        activated.add(node_id)
        chosen_nodes.append(node_id)
        remaining.difference_update(newly_covered)
        # V8 update_state(): mask each newly covered target column for ALL rows.
        working[:, newly_covered] = LARGE_VALUE
    if remaining:
        raise RuntimeError(f"Exceeded {MAX_STEPS} actions with uncovered nodes {sorted(remaining)}")
    return chosen_nodes, accumulated_cost


def predict_one(json_path, process_topology, actor1, actor2, device):
    # Same ingestion used for training; DO NOT substitute raw json.load():
    cloud, regional, edge, lat = process_topology(str(json_path))
    latency = np.asarray(lat, dtype=np.float64)
    if latency.shape != (EXPECTED_NODES, EXPECTED_NODES):
        raise ValueError(f"Expected 50x50 matrix, got {latency.shape}")
    if not np.isfinite(latency).all():
        raise ValueError("Latency matrix contains NaN or Inf")
    if np.any(latency < 0):
        raise ValueError("Latency matrix contains negative values")
    node_classes = node_classes_for_topology(cloud, regional, edge, EXPECTED_NODES)
    edge_nodes = np.unique(np.atleast_1d(edge).astype(int))
    if not len(edge_nodes):
        raise ValueError("Topology has no E2/edge target nodes")

    # Same initialization as V8: mask all non-E2 columns for Agent 1.
    state1 = latency.copy()
    nontarget = np.ones(EXPECTED_NODES, dtype=bool)
    nontarget[edge_nodes] = False
    state1[:, nontarget] = LARGE_VALUE

    nodes1, cost1 = select_nodes(
        actor1, state1, edge_nodes, VARIABLE_COSTS_1, node_classes, device
    )
    selected1 = list(dict.fromkeys(nodes1))

    # Same V8 E2 -> nearest selected G1 association (on original latency).
    e2_to_g1 = {}
    first_stage = {}
    for e2 in edge_nodes.tolist():
        g1 = min(selected1, key=lambda g: float(latency[g, e2]))
        hop1 = float(latency[g1, e2])
        if hop1 > LATENCY_LIMIT_MS:
            raise RuntimeError(f"E2 {e2}: nearest selected G1 {g1} has {hop1} ms > 10 ms")
        e2_to_g1[e2] = g1
        first_stage[e2] = hop1

    # Same V8 worst-first-hop-per-G1 aggregation and Agent 2 input format.
    g1_to_e2 = {}
    for e2, g1 in e2_to_g1.items():
        g1_to_e2.setdefault(g1, []).append(e2)
    active_g1 = sorted(g1_to_e2)
    worst_first_hop = {g: max(first_stage[e] for e in e2s)
                       for g, e2s in g1_to_e2.items()}
    state2 = np.full((EXPECTED_NODES, EXPECTED_NODES), LARGE_VALUE, dtype=np.float64)
    for g1 in active_g1:
        state2[:, g1] = latency[:, g1] + worst_first_hop[g1]
    for g1 in active_g1:
        if not np.any(state2[:, g1] <= LATENCY_LIMIT_MS):
            raise RuntimeError(f"G1 {g1} has no candidate G2 with complete-loop latency <=10 ms")

    # Reuse G1 physical nodes without paying their fixed costs again.
    nodes2, cost2 = select_nodes(
        actor2, state2, active_g1, VARIABLE_COSTS_2,
        node_classes, device, preactivated_nodes=selected1
    )
    selected2 = list(dict.fromkeys(nodes2))

    # Same final G1 -> feasible selected G2 association and individual loop check.
    g1_to_g2 = {}
    for g1 in active_g1:
        feasible = [g2 for g2 in selected2
                    if worst_first_hop[g1] + latency[g2, g1] <= LATENCY_LIMIT_MS]
        if not feasible:
            raise RuntimeError(f"G1 {g1} has no feasible selected G2")
        g1_to_g2[g1] = min(feasible,
                           key=lambda g2: float(worst_first_hop[g1] + latency[g2, g1]))
    max_loop = 0.0
    for e2, g1 in e2_to_g1.items():
        loop_ms = first_stage[e2] + float(latency[g1_to_g2[g1], g1])
        if loop_ms > LATENCY_LIMIT_MS:
            raise RuntimeError(f"E2 {e2} -> G1 {g1} -> G2 {g1_to_g2[g1]}: "
                               f"{loop_ms} ms exceeds 10 ms")
        max_loop = max(max_loop, loop_ms)

    return {
        "status": "OK",
        "cost1": cost1,
        "cost2": cost2,
        "cost_total": cost1 + cost2,
        "computing_nodes_list_1": nodes1,
        "computing_nodes_list_2": nodes2,
        "e2_to_g1": {str(e): int(g) for e, g in e2_to_g1.items()},
        "g1_to_g2": {str(g): int(g2) for g, g2 in g1_to_g2.items()},
        "max_complete_loop_latency_ms": max_loop,
        "num_agent1_actions": len(nodes1),
        "num_agent2_actions": len(nodes2),
    }

## 2. Read all topology JSON files and load the saved models

In [3]:
# Load the same JSON parser used by the training notebook.
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"PROJECT_DIR does not exist: {PROJECT_DIR}")
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

sys.path.insert(0, str(PROJECT_DIR))
try:
    from data_extract_50_updated import process_topology
except ImportError as exc:
    raise ImportError(
        "Cannot import data_extract_45_updated.process_topology. "
        "Place the SAME data_extract_45_updated.py used during training "
        "in PROJECT_DIR or make it importable."
    ) from exc

def topology_index(path):
    match = re.search(r"_(\d+)\.json$", path.name)
    return (int(match.group(1)) if match else float("inf"), path.name)

topology_files = sorted(PROJECT_DIR.glob(TOPOLOGY_PATTERN), key=topology_index)
if not topology_files:
    raise FileNotFoundError(
        f"No JSON files match {TOPOLOGY_PATTERN!r} in {PROJECT_DIR}"
    )

# This checkpoint already contains the learned weights: NO new training.
torch.set_num_threads(min(torch.get_num_threads(), 4))
actor1, actor2 = load_actors(CHECKPOINT_PATH, DEVICE)

print(f"Loaded trained actors; discovered {len(topology_files)} JSON topologies.")
print("Inference mode: deterministic argmax among the V8-valid actions.")
print("First five JSONs:", [p.name for p in topology_files[:5]])

Loaded trained actors; discovered 20 JSON topologies.
Inference mode: deterministic argmax among the V8-valid actions.
First five JSONs: ['50_CNs_topology_0.json', '50_CNs_topology_5.json', '50_CNs_topology_10.json', '50_CNs_topology_15.json', '50_CNs_topology_20.json']


## 3. Predict on each JSON; show costs, both computing-node lists, and unique nodes

In [4]:
# Execute Agent 1 -> Agent 2 predictions independently for EVERY JSON.
# A failed JSON is reported as ERROR; it is never silently repaired or
# treated as a feasible placement. Each success row displays the full lists.
all_predictions = []
summary_rows = []

for path in topology_files:
    try:
        prediction = predict_one(path, process_topology, actor1, actor2, DEVICE)

        nodes1 = [int(n) for n in prediction["computing_nodes_list_1"]]
        nodes2 = [int(n) for n in prediction["computing_nodes_list_2"]]
        unique1 = sorted(set(nodes1))
        unique2 = sorted(set(nodes2))
        unique_joint = sorted(set(nodes1) | set(nodes2))
        shared = sorted(set(nodes1) & set(nodes2))

        prediction.update({
            "unique_nodes_list_1": unique1,
            "unique_nodes_list_2": unique2,
            "unique_nodes_list_combined": unique_joint,
            "shared_nodes_list": shared,
            "num_unique_nodes_combined": len(unique_joint),
        })
        print(f"\n{path.name}: OK")
        print(f"  Agent 1: cost1={prediction['cost1']:g}; nodes={nodes1}; unique={unique1}")
        print(f"  Agent 2: cost2={prediction['cost2']:g}; nodes={nodes2}; unique={unique2}")
        print(f"  Combined: total_cost={prediction['cost_total']:g}; "
              f"unique_nodes={unique_joint}; shared_nodes={shared}")
        print(f"  Max E2 -> G1 -> G2 latency: "
              f"{prediction['max_complete_loop_latency_ms']:g} ms")
    except Exception as exc:
        prediction = {
            "status": "ERROR",
            "error": f"{type(exc).__name__}: {exc}",
            "cost1": None, "cost2": None, "cost_total": None,
            "computing_nodes_list_1": [], "computing_nodes_list_2": [],
            "unique_nodes_list_1": [], "unique_nodes_list_2": [],
            "unique_nodes_list_combined": [], "shared_nodes_list": [],
            "num_unique_nodes_combined": None,
            "max_complete_loop_latency_ms": None,
        }
        print(f"\n{path.name}: ERROR: {prediction['error']}")

    detail = {"topology_file": path.name, **prediction}
    all_predictions.append(detail)
    summary_rows.append({
        "topology_file": path.name,
        "status": prediction["status"],
        "cost1": prediction["cost1"],
        "cost2": prediction["cost2"],
        "cost_total": prediction["cost_total"],
        "computing_nodes_list_1": prediction["computing_nodes_list_1"],
        "computing_nodes_list_2": prediction["computing_nodes_list_2"],
        "unique_nodes_list_1": prediction["unique_nodes_list_1"],
        "unique_nodes_list_2": prediction["unique_nodes_list_2"],
        "unique_nodes_list_combined": prediction["unique_nodes_list_combined"],
        "shared_nodes_list": prediction["shared_nodes_list"],
        "num_unique_nodes_combined": prediction["num_unique_nodes_combined"],
        "max_complete_loop_latency_ms": prediction["max_complete_loop_latency_ms"],
        "error": prediction.get("error", ""),
    })

# Render ONE row per JSON in the notebook, including all node-list columns.
results_df = pd.DataFrame(summary_rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
print("\n\n=== ALL TOPOLOGIES: AGENT 1 / AGENT 2 / UNIQUE NODES ===")
display(results_df)
print(f"Feasible predictions: "
      f"{sum(r['status'] == 'OK' for r in all_predictions)} / {len(all_predictions)}")


50_CNs_topology_0.json: OK
  Agent 1: cost1=40; nodes=[9, 2]; unique=[2, 9]
  Agent 2: cost2=16; nodes=[9, 2]; unique=[2, 9]
  Combined: total_cost=56; unique_nodes=[2, 9]; shared_nodes=[2, 9]
  Max E2 -> G1 -> G2 latency: 10 ms

50_CNs_topology_5.json: OK
  Agent 1: cost1=60; nodes=[9, 2, 1]; unique=[1, 2, 9]
  Agent 2: cost2=25; nodes=[0, 9, 2]; unique=[0, 2, 9]
  Combined: total_cost=85; unique_nodes=[0, 1, 2, 9]; shared_nodes=[2, 9]
  Max E2 -> G1 -> G2 latency: 10 ms

50_CNs_topology_10.json: OK
  Agent 1: cost1=60; nodes=[9, 2, 1]; unique=[1, 2, 9]
  Agent 2: cost2=25; nodes=[0, 9, 2]; unique=[0, 2, 9]
  Combined: total_cost=85; unique_nodes=[0, 1, 2, 9]; shared_nodes=[2, 9]
  Max E2 -> G1 -> G2 latency: 10 ms

50_CNs_topology_15.json: OK
  Agent 1: cost1=40; nodes=[9, 2]; unique=[2, 9]
  Agent 2: cost2=16; nodes=[9, 2]; unique=[2, 9]
  Combined: total_cost=56; unique_nodes=[2, 9]; shared_nodes=[2, 9]
  Max E2 -> G1 -> G2 latency: 10 ms

50_CNs_topology_20.json: OK
  Agent 1: co

,topology_file,status,cost1,cost2,cost_total,computing_nodes_list_1,computing_nodes_list_2,unique_nodes_list_1,unique_nodes_list_2,unique_nodes_list_combined,shared_nodes_list,num_unique_nodes_combined,max_complete_loop_latency_ms,error
0,50_CNs_topology_0.json,OK,40.0,16.0,56.0,"[9, 2]","[9, 2]","[2, 9]","[2, 9]","[2, 9]","[2, 9]",2,10.0,
1,50_CNs_topology_5.json,OK,60.0,25.0,85.0,"[9, 2, 1]","[0, 9, 2]","[1, 2, 9]","[0, 2, 9]","[0, 1, 2, 9]","[2, 9]",4,10.0,
2,50_CNs_topology_10.json,OK,60.0,25.0,85.0,"[9, 2, 1]","[0, 9, 2]","[1, 2, 9]","[0, 2, 9]","[0, 1, 2, 9]","[2, 9]",4,10.0,
3,50_CNs_topology_15.json,OK,40.0,16.0,56.0,"[9, 2]","[9, 2]","[2, 9]","[2, 9]","[2, 9]","[2, 9]",2,10.0,
4,50_CNs_topology_20.json,OK,60.0,52.0,112.0,"[9, 2, 11]","[9, 2, 43]","[2, 9, 11]","[2, 9, 43]","[2, 9, 11, 43]","[2, 9]",4,10.0,
5,50_CNs_topology_25.json,OK,60.0,16.0,76.0,"[9, 2, 1]","[9, 1]","[1, 2, 9]","[1, 9]","[1, 2, 9]","[1, 9]",3,10.0,
6,50_CNs_topology_30.json,OK,60.0,24.0,84.0,"[9, 2, 1]","[9, 2, 1]","[1, 2, 9]","[1, 2, 9]","[1, 2, 9]","[1, 2, 9]",3,10.0,
7,50_CNs_topology_35.json,OK,20.0,8.0,28.0,[9],[9],[9],[9],[9],[9],1,10.0,
8,50_CNs_topology_40.json,OK,60.0,52.0,112.0,"[9, 2, 1]","[9, 37, 1]","[1, 2, 9]","[1, 9, 37]","[1, 2, 9, 37]","[1, 9]",4,10.0,
9,50_CNs_topology_45.json,OK,40.0,16.0,56.0,"[9, 2]","[9, 2]","[2, 9]","[2, 9]","[2, 9]","[2, 9]",2,10.0,


Feasible predictions: 20 / 20


## 4. Save predictions to CSV and JSON

The CSV contains one row per JSON. The detailed JSON file additionally stores the E2-to-G1 and G1-to-G2 associations and feasibility checks.

In [5]:
# Export one summary row per topology, plus complete association details.
csv_path = OUTPUT_PREFIX.with_suffix(".csv")
json_path = OUTPUT_PREFIX.with_suffix(".json")
csv_path.parent.mkdir(parents=True, exist_ok=True)

# Encode list-valued columns as JSON strings in the CSV so they can be reloaded.
results_csv = results_df.copy()
for column in results_csv.columns:
    if column.endswith("_list_1") or column.endswith("_list_2") or column in (
        "unique_nodes_list_combined", "shared_nodes_list"
    ):
        results_csv[column] = results_csv[column].map(json.dumps)
results_csv.to_csv(csv_path, index=False)
json_path.write_text(json.dumps(all_predictions, indent=2), encoding="utf-8")

print(f"Saved: {csv_path}")
print(f"Saved: {json_path}")
print("Notebook table remains available as results_df.")

Saved: /home/cci/Downloads/Latest/Revised/Time_Vary_50N/ha2c_v8_50_topology_predictions.csv
Saved: /home/cci/Downloads/Latest/Revised/Time_Vary_50N/ha2c_v8_50_topology_predictions.json
Notebook table remains available as results_df.
